In [ ]:
from arch import arch_model
import numpy as np
import pandas as pd
from scipy.stats import t as t_dist

# =========================================
# VaR GARCH(1,1) con distribución t-Student
# =========================================
# Idéntico al modelo GARCH-Normal pero con distribución de errores t-Student.
# La t-Student tiene colas más pesadas que la normal, lo que permite capturar
# mejor los eventos extremos (fat tails) habituales en los mercados financieros.
#
# Los grados de libertad ν se estiman junto con los parámetros GARCH:
# valores bajos de ν (típicamente 4-8) indican colas muy pesadas.
#
# Configuración (igual que MLP para comparación justa):
#   - Ventana de entrenamiento: 900 observaciones
#   - Reentrenamiento: diario (parámetros reestimados cada sesión)
# =========================================

# Cargamos el dataset preprocesado (generado en datos.ipynb)
dataset = pd.read_pickle("dataset_tfg.pkl").copy().sort_index()

alpha = 0.99       # nivel de confianza del VaR (99%)
retrain_every = 1  # reestimamos los parámetros GARCH cada día

eval_start = pd.Timestamp("2015-01-01")
eval_end   = pd.Timestamp("2024-12-31")
eval_dates = dataset.loc[eval_start:eval_end].index

var_preds = []
real_targets = []
dates = []
current_model = None  # almacena el último modelo GARCH válido estimado

print("Iniciando GARCH-t rolling...\n")

for i, current_date in enumerate(eval_dates):

    if i % retrain_every == 0:
        # Seleccionamos las 900 observaciones anteriores al día actual
        train_end = current_date - pd.Timedelta(days=1)
        train_df = dataset.loc[:train_end].tail(900)

        # Retornos en porcentaje para la librería arch
        r_train = (train_df["rp_lag_0"].dropna() * 100).astype(float)

        if len(r_train) >= 250:
            # Especificación: media constante, volatilidad GARCH(1,1), errores t-Student
            # dist="t" hace que el modelo estime también los grados de libertad ν
            am = arch_model(
                r_train,
                mean="Constant",
                vol="GARCH",
                p=1, q=1,
                dist="t"
            )
            try:
                # Estimación por máxima verosimilitud; disp="off" suprime output
                current_model = am.fit(disp="off")
            except Exception:
                pass   # si la estimación no converge, mantenemos el último modelo válido

    if current_model is None:
        continue

    try:
        # Predicción de la varianza condicional para el siguiente día (horizonte h=1)
        forecast = current_model.forecast(horizon=1, reindex=False)
        mu_t    = forecast.mean.iloc[-1, 0]               # media condicional estimada
        sigma_t = np.sqrt(forecast.variance.iloc[-1, 0])  # desv. típica condicional

        # Grados de libertad ν estimados por máxima verosimilitud
        # Valores bajos → colas más pesadas → VaR más conservador
        nu = current_model.params.get("nu", 10.0)

        # Cuantil de la t-Student al nivel (1 - alpha) con ν grados de libertad
        z  = t_dist.ppf(1 - alpha, df=nu)

        # VaR expresado como pérdida positiva: negamos y convertimos de % a fracción
        var_return = mu_t + sigma_t * z
        var_loss_pred = -var_return / 100.0

        real_loss = dataset.loc[current_date, "target"]

        var_preds.append(var_loss_pred)
        real_targets.append(real_loss)
        dates.append(current_date)

    except Exception:
        continue  # si la predicción falla, omitimos ese día

# ==============================
# Resultados y métricas básicas
# ==============================
out_garch = pd.DataFrame(
    {"VaR_GARCH": np.array(var_preds), "loss_real": np.array(real_targets)},
    index=pd.to_datetime(dates)
).sort_index()

out_garch = out_garch.loc["2015":"2024"].dropna().copy()

# Indicador de violación: 1 si la pérdida real supera el VaR estimado
out_garch["viol"] = (out_garch["loss_real"] > out_garch["VaR_GARCH"]).astype(int)
out_garch["year"] = out_garch.index.year

# Tasa de violación: una tasa < 1% indica conservadurismo (VaR sobreestimado)
viol_rate = out_garch["viol"].mean()
yearly = out_garch.groupby("year")["viol"].mean()

print("\n===================================")
print("VaR GARCH(1,1) t-Student")
print("===================================")
print(f"Ventana         : 900 obs")
print(f"Retrain cada    : {retrain_every} día(s)")
print(f"Total preds     : {len(out_garch)}")
print(f"Violation rate  : {viol_rate:.4f}")
print(f"Esperado        : {1-alpha:.4f}")

print("\nViolaciones por año:")
for y, v in yearly.items():
    print(f"{y}: {v:.4f}")

# Exportamos para el análisis comparativo en comp_final.ipynb
out_garch.to_csv("garch_t_var_predictions.csv")
print("\nGuardado: garch_t_var_predictions.csv")